In [2]:
!pip install -q -U langchain langchain-community langchain-google-genai faiss-cpu pypdf

In [3]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"]=userdata.get("GEMINI_API_KEYS")
MODEL="gemini-3.5-flash-lite"
EMB_MODEL="gemini-embedding-001"

In [58]:
#1.Prepare the document]
#1.1 Upload document
from google.colab import files
uploaded=files.upload()
pdf_name=list(uploaded.keys())[0]
print(f"PDF:{pdf_name} is uploaded sucessfully")

Saving 5-Day Executive Meeting Schedule.pdf to 5-Day Executive Meeting Schedule (2).pdf
PDF:5-Day Executive Meeting Schedule (2).pdf is uploaded sucessfully


In [59]:
#1.2 extract text from document
from langchain_community.document_loaders import PyPDFLoader
docs=PyPDFLoader(pdf_name).load()
print(f"DOCUMENT WITH {len(docs) is uploaded} sucessfully")

text=""
for d in docs:
  text+=d.page_content+"\n-------------------\n"

print("No of characters in document: ",len(text))
print("First few contents of page 1: ",docs[0].page_content[:100])
print("META DATA OF PAGE 1: ",docs[0].metadata)

DOCUMENT WITH False sucessfully
No of characters in document:  3431
First few contents of page 1:  Executive Strategic Week: Master Schedule
COMPREHENSIVE 5-DAY INTERDEPARTMENTAL BRIEFING PROGRAM
Ove
META DATA OF PAGE 1:  {'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-09-23T17:54:33+00:00', 'moddate': '2026-09-23T17:54:33+00:00', 'source': '5-Day Executive Meeting Schedule (2).pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


In [60]:
#1.3 Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
chunks=splitter.split_documents(docs)
print("No of chunks: ",len(chunks))
print(chunks[1])

No of chunks:  9
page_content='Date & Time: Oct 12, 2026 | 09:00 - 11:30  Venue: Global Boardroom, Floor 14
Agenda: Q3 KPI reviews, strategic realignment, and roadmap approvals.  Attendees: Executive Board, Managing
Directors
Description: The week commences with an exhaustive examination of our operational performance metrics from the
previous quarter. Leadership will align on core strategic priorities, capital allocation, and long-term milestones
designed to sustain our competitive market advantage.' metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-09-23T17:54:33+00:00', 'moddate': '2026-09-23T17:54:33+00:00', 'source': '5-Day Executive Meeting Schedule (2).pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


In [61]:
#1.4Embeddings vectorstore
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
embeddings=GoogleGenerativeAIEmbeddings(model=EMB_MODEL)
vectorstore=FAISS.from_documents(chunks,embeddings)

In [62]:
#1.4 Retriever
retriever=vectorstore.as_retriever(search_kwargs={'k':6})
print(retriever.invoke("Tell me which meeting is on october 16 2026"))

[Document(id='926b3fda-0314-4254-b76c-d4aab043cf87', metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-09-23T17:54:33+00:00', 'moddate': '2026-09-23T17:54:33+00:00', 'source': '5-Day Executive Meeting Schedule (2).pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Date & Time: Oct 12, 2026 | 09:00 - 11:30  Venue: Global Boardroom, Floor 14\nAgenda: Q3 KPI reviews, strategic realignment, and roadmap approvals.  Attendees: Executive Board, Managing\nDirectors\nDescription: The week commences with an exhaustive examination of our operational performance metrics from the\nprevious quarter. Leadership will align on core strategic priorities, capital allocation, and long-term milestones\ndesigned to sustain our competitive market advantage.'), Document(id='a0951a37-9098-4033-90d8-80c78ae1d495', metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-09-23T17:54:33+00:00', 'moddate': '2026-09-23T17:54:33+00:00', '

In [67]:
from pydantic import BaseModel
from typing import Optional
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm=ChatGoogleGenerativeAI(model=MODEL,max_output_tokens=1500)
class Schema(BaseModel):
  date:Optional[str]=None
  day:Optional[str]=None
  venue:Optional[str]=None
  event:Optional[str]=None
  agenda:Optional[str]=None
str_llm=llm.with_structured_output(Schema)

prompt=prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a meeting information extraction assistant.

Your task is to extract information ONLY about the meeting/event
requested in the QUESTION from the provided CONTEXT.

Extract these fields:
- date
- day
- venue
- event
- agenda
- description

IMPORTANT RULES:

1. First identify the EXACT meeting/event in the CONTEXT that matches
   the QUESTION.

2. Once the matching meeting is identified, extract ONLY information
   belonging to that meeting.

3. DATE:
   Extract the date/date-time belonging to the matching meeting.

4. DAY:
   Extract the day belonging to the matching meeting.

5. VENUE:
   Extract the venue belonging to the matching meeting.

6. EVENT:
   Extract the meeting/event title from the matching meeting heading.

7. AGENDA:
   Extract the text labelled "Agenda" belonging to the matching meeting.

8. DESCRIPTION:
   The description belongs to the matching meeting/event.

   If the matching meeting has an explicitly labelled "Description:",
   extract that description.

   If the matching meeting does NOT have a "Description:" label,
   identify the unlabelled paragraph immediately associated with that
   meeting and use that paragraph as the description.

9. VERY IMPORTANT:
   Do NOT use a "Description:" belonging to another section, another
   meeting, administrative notes, or any unrelated content.

10. A description must be associated with the matching meeting.
    Do not select a description merely because it appears later in
    the CONTEXT.

11. If multiple meetings or descriptions appear in the CONTEXT,
    use only the information belonging to the meeting requested
    in the QUESTION.

12. Do not combine information from different meetings or sections.

13. Do not invent or guess information.

14. If a field genuinely does not exist for the requested meeting,
    return null.

Return the extracted information in the structured format.
"""),

    ("human", """
CONTEXT:
{context}

QUESTION:
{question}
""")
])

def format_docs(docs):
  return "\n----------\n".join(d.page_content for d in docs)
rag_chain=(
    {'context':retriever | format_docs,'question':RunnablePassthrough()}
    |prompt
    |str_llm

)

In [74]:
print("Enter Exit,Bye,Quit to terminate session")
while True:
  user_input=input("Enter your meeting query: ")
  if user_input.lower() in ['quit','bye','exit']:
    print("Exting......\nExited")
    break
  result=rag_chain.invoke(user_input)

  result=rag_chain.invoke("List me meeting is on october 16 2026")

  display_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a meeting assistant.

You are given structured information about a meeting and the
user's question.

Your job is to display ONLY the information requested by the user.

Rules:

- If the user asks for the date, display only the date.
- If the user asks for the day, display only the day.
- If the user asks for the venue, display only the venue.
- If the user asks for the event, display only the event.
- If the user asks for the agenda, display only the agenda.
- If the user asks for the description, display only the description.
- If the user asks for multiple fields, display only those fields.
- If the user asks for all information/details, display all
  available fields.
- Do not display fields that the user did not ask for.
- Do not invent information.
return answer in proper sentence , properly form sentence

MEETING INFORMATION:
{meeting_info}

USER QUESTION:
{question}
"""),

    ("human", "{question}")
  ])

  display_chain=display_prompt | llm
  meeting_info = f"""
  DATE: {result.date}
  DAY: {result.day}
  VENUE: {result.venue}
  EVENT: {result.event}
  AGENDA: {result.agenda}

  """
  response=display_chain.invoke({"meeting_info":meeting_info,"question":user_input})
  print(response.text)


Enter Exit,Bye,Quit to terminate session
Enter your meeting query: tell me date of meeting for cto
The date of the meeting is Oct 16, 2026.
Enter your meeting query: what is agenda for meeting on thursday can u tell me date
The date of the meeting is Oct 16, 2026, and the agenda is weekly findings synthesis, accountability frameworks, closing notes.
Enter your meeting query: exit
Exting......
Exited
